# Day 2, afternoon: adding language information to CTC

The same kind of **workbench**: you write no code here. Each section calls the package and draws what comes
out. The figures follow lecture L4 (and L3's CTC section).

| You implement | In | Slides | Sections here |
|---|---|---|---|
| `greedy_ctc` | `speech/ctc_lm.py` | L3, "Greedy decoding" | 2, 3 |
| `sweep` | `speech/ctc_lm.py` | L4, "`sweep`: what to implement" | 5 |

Both functions are written except a few lines (2 holes each): every
`raise NotImplementedError("TODO(d2): ...")` marks a hole and says what to write there.

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:10 | Section 1 (provided): the CTC output of `wav2vec2-base-100h` on 40 utterances | about 30 s |
| 0:10–0:30 | `greedy_ctc` (2 holes); sections 2–3; `pytest -m d2 -k greedy` | |
| 0:30–0:45 | Section 4: the LM on one utterance, its two knobs | a few s |
| 0:45–1:10 | `sweep` (2 holes); section 5; `pytest -m d2` | about 1 min |
| 1:10–1:15 | Start the graded run in a terminal (below) | about 2 min |
| 1:15–1:40 | Section 6: your graded rows; the questions | |
| 1:40–2:00 | One `DECISIONS.md` entry; tag `d2` | |

The graded run (alpha × beta × beam tuned on 100 dev utterances, the best point decoded once on 100 others):
```
export DSP_MILESTONE=d2          # PowerShell: $env:DSP_MILESTONE = "d2"
python -m speech.ctc_lm --arpa data/lm/3-gram.pruned.1e-7.arpa
```
It needs `pyctcdecode` and `kenlm` (`uv sync --extra lm`, README "Setup"), and the LM from
`python scripts/fetch_data.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root
DATA = Path(os.environ.get("DSP_DATA", ROOT / "data"))
ARPA = DATA / "lm/3-gram.pruned.1e-7.arpa"

import pandas as pd
from IPython.display import display

from speech.ctc_lm import greedy_ctc, librispeech_logits, make_decoder, sweep
from speech.data import wer

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
try:
    import kenlm  # noqa: F401
    import pyctcdecode  # noqa: F401
    HAVE_DECODER = True
except ImportError as e:
    HAVE_DECODER = False
    print(f"{e.name} is not installed: sections 3-6 need it (`uv sync --extra lm`, README 'Setup').")
if not ARPA.exists():
    print(f"No {ARPA}: run `python scripts/fetch_data.py` (sections 4-6 need the LM).")


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")

## 1. The CTC output, computed once

`wav2vec2-base-100h`: fine-tuned on 100 h of transcripts only, so it leaves errors for a language model to fix
(L3, "wav2vec 2.0: the three checkpoints"). The provided `librispeech_logits` runs it once on 40 utterances and
returns, for each, `logprobs` of shape $T\times32$; every decoder below reads these arrays, the network never
runs again. `labels[i]` is the token of id `i` (the blank is `''`, the word delimiter `' '`).

In [ ]:
lps, refs, labels, blank = librispeech_logits(DATA / "LibriSpeech", "facebook/wav2vec2-base-100h", 40, "cpu")
print(f"{len(lps)} utterances; first: logprobs {lps[0].shape}, blank id {blank}")
print("labels:", labels)

## 2. Your `greedy_ctc`: text, not words

The best label at every frame, then the collapse $\mathcal B$ (L3, "Greedy decoding"). Below, three
utterances, the WER over 40, and the words of your output that the LibriSpeech LM has never seen
(out of its 200 000-word vocabulary): misspellings that CTC alone cannot notice.

In [ ]:
def lm_vocabulary(arpa):
    """The unigrams of an ARPA file (read until the bigram section)."""
    words, inside = set(), False
    with open(arpa, encoding="utf-8") as f:
        for line in f:
            if line.startswith("\\1-grams:"):
                inside = True
            elif line.startswith("\\2-grams:"):
                break
            elif inside and line.strip():
                words.add(line.split("\t")[1].lower())
    return words


greedy = None
try:
    greedy = [greedy_ctc(lp, labels, blank) for lp in lps]
except NotImplementedError as e:
    not_yet(e)
if greedy is not None:
    for r, h in list(zip(refs, greedy))[:3]:
        print(f"REF {r}\nHYP {h}\n")
    print(f"greedy WER over {len(refs)} utterances: {wer(refs, greedy):.3f}")
    if ARPA.exists():
        vocab = lm_vocabulary(ARPA)
        non_words = sorted({w for h in greedy for w in h.split() if w not in vocab})
        print(f"{len(non_words)} output words unknown to the LM, e.g.:", ", ".join(non_words[:15]))

## 3. Prefix beam search on L4's toy example

The four frames of L4, "Prefix beam search, frame by frame": tokens ∅, `a`, `b`. Greedy keeps the best label of
each frame; `pyctcdecode`'s prefix beam search keeps the $N_{beam}$ best prefixes with their summed
probabilities. The slide says: greedy and $N_{beam}=1$ return `a`, $N_{beam}=2$ returns `aa` (exact
probability 0.528 against 0.372).

In [ ]:
P = np.array([[0.5, 0.4, 0.1], [0.6, 0.4, 0.0], [0.8, 0.2, 0.0], [0.0, 1.0, 0.0]])  # rows: frames; columns: ∅, a, b
toy = np.log(np.maximum(P, 1e-12)).astype(np.float32)
try:
    print("your greedy_ctc:", repr(greedy_ctc(toy, ["", "a", "b"], 0)))
except NotImplementedError as e:
    not_yet(e)
if HAVE_DECODER:
    from pyctcdecode import build_ctcdecoder
    toy_dec = build_ctcdecoder(["", "a", "b"])
    for n in (1, 2, 3):  # no pruning other than the beam itself
        print(f"prefix beam search, N_beam={n}:",
              repr(toy_dec.decode(toy, beam_width=n, beam_prune_logp=-1e9, token_min_logp=-1e9)))

## 4. The LM on one utterance: $\lambda$ (`alpha`) and $\beta$ (`beta`)

Shallow fusion ranks each prefix by $\log P_{\mathrm{ctc}} + \lambda\log P_{\mathrm{LM}} + \beta|W|$
(L4, "Shallow fusion: the day-1 formula again"). The first utterance of section 2, decoded with a beam of 50:
without LM, then with the LM at several $(\lambda, \beta)$.

In [ ]:
if HAVE_DECODER and ARPA.exists() and greedy is not None:
    i = 0
    dec = make_decoder(labels, str(ARPA), 0.5, 1.0)
    print(f"REF                  {refs[i]}\ngreedy               {greedy[i]}")
    print(f"beam 50, no LM       {make_decoder(labels, None, 0, 0).decode(lps[i], beam_width=50)}")
    for a, b in ((0.5, 1.0), (0.5, -5.0), (2.0, 1.0), (5.0, 1.0), (10.0, 1.0)):
        dec.reset_params(alpha=a, beta=b)
        print(f"alpha={a:<4} beta={b:<5}{dec.decode(lps[i], beam_width=50)}")
elif greedy is None:
    print("Skipped: this cell needs your greedy_ctc (section 2).")

**Questions.** Which words does a small $\lambda$ fix? With a large $\lambda$ the output glues words together
(`trainingof`): every word pays $\lambda\log P_{\mathrm{LM}}<0$, so fewer words cost less. Which term of the
score is there to pay that back, and on which data would you choose both values?

## 5. Your `sweep`: greedy, beam, beam + LM

Your `sweep` decodes every utterance with each configuration and returns one row per configuration. Tuning
happens on a **dev** half (the first 20 utterances); the best point is decoded once on the **test** half
(L4, "Tuning protocol, and what the self-test shows"). The figure: WER against decoding time per utterance, every row of dev.

In [ ]:
dev_rows = None
if HAVE_DECODER and ARPA.exists():
    try:
        dev_rows = pd.DataFrame(sweep(lps[:20], refs[:20], labels, blank, str(ARPA), alphas=[0.5, 1.0, 2.0],
                                      betas=[0.0, 1.0], beams=[10, 50]))
    except NotImplementedError as e:
        not_yet(e)
if dev_rows is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in dev_rows.groupby("decoder", sort=False):
        ax.plot(g.ms_per_utt, 100 * g.wer, "o", label=name)
    ax.set(xscale="log", xlabel="decoding time per utterance (ms)", ylabel="dev WER (%)", title="20 dev utterances")
    ax.legend(fontsize=8)
    fig.tight_layout()
    best = dev_rows[dev_rows.decoder != "greedy (acoustic only)"].sort_values("wer").iloc[0]
    print("best on dev:", best.to_dict())
    test_rows = pd.DataFrame(sweep(lps[20:], refs[20:], labels, blank,
                                   str(ARPA) if best.decoder == "beam + n-gram" else None,
                                   alphas=[best.alpha], betas=[best.beta], beams=[int(best.beam)]))
    print("test, decoded once with the dev choice:")
    display(test_rows.round(3))

**Questions.** How much of the gain comes from the search alone (beam, no LM) and how much from the LM? What
does the LM cost in decoding time? Would you trust a difference of 1 point of WER on 20 utterances?

## 6. Your graded rows

After the graded run (top of the notebook), this cell reads the `ctc_decoding` rows of
`results/results.jsonl`: the test WER of each decoder (100 utterances), and the dev sweep.

In [ ]:
from common.bench import load_records

res = ROOT / "results/results.jsonl"
df = load_records(res) if res.exists() else None
if df is None or "task" not in df or (df.task == "ctc_decoding").sum() == 0:
    print("No 'ctc_decoding' rows yet: run the graded command above, then re-run this cell.")
else:
    df = df[df.task == "ctc_decoding"].copy()
    df["decoder"] = df.model.str.split("|").str[-1]
    test = df[df.cfg_split == "test"]
    display(test[["decoder", "cfg_alpha", "cfg_beta", "cfg_beam", "wer", "ms_per_utt"]].round(4))
    devd = df[df.cfg_split == "dev"]
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in devd.groupby("decoder", sort=False):
        ax.plot(g.ms_per_utt, 100 * g.wer, "o", alpha=0.6, label=name)
    ax.set(xscale="log", xlabel="decoding time per utterance (ms)", ylabel="dev WER (%)", title="graded dev sweep")
    ax.legend(fontsize=8)
    fig.tight_layout()

## Before you leave

Write at least one `DECISIONS.md` entry tagged d2: explicit LM (this afternoon) or implicit LM (Whisper, this
morning) for a push-to-talk assistant, and which $N_{beam}$, with the rows that justify it.